# Study 816 — Drawdown Duration ⏱️📉

**Does how *long* a name spent underwater over the past year predict its future return?**

A drawdown has two moments: its **depth** (how far a name fell) and its **duration** (how
long it stayed down). We take the duration side as **time-underwater** — the fraction of
the trailing year a name's cumulative total return sat **below its running high-water
mark** — and ask whether the market **pays** for bearing persistent-drawdown names (they
rebound), or whether they simply **keep sinking**. Liquid US cross-section
(2010-01-04 → 2026-06-30, 50 names). Honest sign.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — the names that stayed
underwater and died are absent, so any "losers keep sinking" tilt is understated.*


## 1. The idea in one picture

Track a name's cumulative total return and its **high-water mark** (the highest level it has reached so far). Whenever the curve is *below* that mark, the name is **underwater**. Add up the underwater days over the last year and divide by the year: that fraction is **time-underwater**. A name always making fresh highs is near 0; a persistent laggard is near 1. Does that persistent-drawdown risk earn a premium — or keep sinking?

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-1.12, t_nw=-0.8, hi_bps=6.62, lo_bps=7.75, gross_sharpe=-0.2)
print('long high-underwater / short low-underwater spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  high-underwater book %+.2f bps vs low-underwater book %+.2f bps'
      % (R['hi_bps'], R['lo_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

long high-underwater / short low-underwater spread: -1.12 bps/day (NW t = -0.80)
  high-underwater book +6.62 bps vs low-underwater book +7.75 bps
  gross spread Sharpe (before cost): -0.20


## 2. A tiny time-underwater example

A price that rises to a peak then drifts below it is underwater exactly on the days strictly beneath its running high-water mark. This is the whole signal, vectorised: `cumprod` → `cummax` → `curve < hwm` → rolling mean.

In [2]:
px = np.array([100,101,102,103,104,103,102,101,100,99], float)
curve = px / px[0]
hwm = np.maximum.accumulate(curve)
uw = (curve < hwm).astype(int)
print('underwater flag per day:', list(uw))
print('time-underwater over these 10 days: %.0f%%' % (100*uw.mean()))

underwater flag per day: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1)]
time-underwater over these 10 days: 50%


## 3. Is the sort just noise? A live synthetic control

We plant the effect in a seeded toy world (`knob>0`: low-drift names stay underwater and keep sinking) and check the detector recovers it — and that it stays *silent* on the null (`knob=0`, time-underwater present but unpriced). No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from drawdown_duration import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(knob=0.0, seed=816, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(knob=0.0010, seed=816, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up, negative)' % planted['t_nw'])

null world   : spread NW t = -0.33  (should be ~0)
planted world: spread NW t = -15.31  (should light up, negative)


## 4. The honest verdict — no signal, no paycheck

On this liquid mega-cap tape the long-high-underwater / short-low-underwater spread is **-1.12 bps/day** with NW *t* = **-0.80** — a statistical **zero**. It sits ≈1.12σ from a 1,000-permutation null (two-sided p = 0.26), and its sign even *flips* between eras (early *t* = +0.28, late *t* = -1.17). The market neither pays a persistent-drawdown premium nor keeps sinking the underwater names. The seeded synthetic control recovers a *planted* relation cleanly, so this flatness is a genuine absence of signal, not a broken sort. **Signal: None**, **Tradability: Mirage** (the book loses to costs either way).